<a href="https://colab.research.google.com/github/voshna123/OODJ_assignment/blob/main/NLP_tf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

--2025-01-13 03:36:20--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py.1’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2025-01-13 03:36:20 (80.9 MB/s) - ‘helper_functions.py.1’ saved [10246/10246]



In [ ]:
from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

In [ ]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip

--2025-01-13 03:36:29--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.216.207, 74.125.26.207, 108.177.11.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.216.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip.1’

nlp_getting_started 100%[===================>] 593.11K  --.-KB/s    in 0.006s  

2025-01-13 03:36:29 (105 MB/s) - ‘nlp_getting_started.zip.1’ saved [607343/607343]



In [ ]:
unzip_data("/content/nlp_getting_started.zip")

In [ ]:
import pandas as pd
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

In [ ]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
# shuffling training dataframe
train_df_shuffled = train_df.sample(frac =1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [ ]:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [ ]:
train_df['target'].value_counts()

,count
target,
0,4342
1,3271


In [ ]:
len(train_df), len(test_df)

(7613, 3263)

In [ ]:
import random

idc = random.randint(0, len(train_df)-5)

train_df_shuffled[['text', 'target']][idc:idc+5]

,text,target
2234,Vince McMahon once again a billionaire: I reme...,0
963,new summer long thin body bag hip A word skirt...,0
4494,HURRICANE GUILLERMO LIVE NOAA TRACKING / LOOPI...,1
2632,We are so arrogant in our destruction that we ...,0
704,@__srajapakse__ Why thank you there missy ?? t...,0


In [ ]:
import sklearn
import tensorflow as tf

X_train, X_test , Y_train, Y_test = sklearn.model_selection.train_test_split(train_df_shuffled['text'].to_numpy(),
                                                                             train_df_shuffled['target'].to_numpy(),
                                                                             test_size = 0.1,
                                                                             random_state = 42)

In [ ]:
len(X_train), len(X_test), len(Y_train), len(Y_test)

(6851, 762, 6851, 762)

In [ ]:
X_train[:10], Y_train[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

## Converting text into numbers

### tokenization

* word-level : I love pizza -> (0) I (1) Love (2) pizza

* character-level : converting letter A-Z to values 1-26

* sub-word tokenization : pizza -> (0) pi (1) zza

In [ ]:
round(sum([len(i.split()) for i in X_train])/len(X_train))

15

In [ ]:
txt_vectoriser = tf.keras.layers.TextVectorization(max_tokens = 10000,
                                                   standardize ='lower_and_strip_punctuation',
                                                   split ='whitespace',
                                                   ngrams = None,
                                                   output_mode = "int",
                                                   output_sequence_length = 15)

In [ ]:
txt_vectoriser.adapt(X_train)

In [ ]:
sample = "lol we vibing"

txt_vectoriser([sample])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[174,  46,   1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [ ]:
random_sentence = random.choice(X_train)

print(f"Original text:\n{random_sentence}\
      \n\nVectorized version:")
txt_vectoriser([random_sentence])

Original text:
Turning rubble from disasters into 'Lego' bricks you can build houses with. This one belongs in #crazyideascollege 
http://t.co/dh0s4bUuK7      

Vectorized version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[3302,  643,   20, 3924,   66, 3682,    1,   12,   71, 1707,  889,
          14,   19,   61, 4100]])>

In [ ]:
words_in_vocab =txt_vectoriser.get_vocabulary()

top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]

top_5_words, bottom_5_words

(['', '[UNK]', 'the', 'a', 'in'],
 ['pages', 'paeds', 'pads', 'padres', 'paddytomlinson1'])

### Embedding

In [ ]:
embedding = tf.keras.layers.Embedding(input_dim=10000,
                                      output_dim = 128,
                                      embeddings_initializer = "uniform",
                                      name = "embedding")

In [ ]:
random_sentence = random.choice(X_train)

print(f"Original text:\n{random_sentence}\
      \n\nVectorized version:")
embedding(txt_vectoriser([random_sentence]))

Original text:
We happily support mydrought  a project bringing awareness to the LA drought. Track your waterÛ_ https://t.co/2ZvhX41I9v      

Vectorized version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[-0.03982311,  0.01599891,  0.03000522, ..., -0.04932341,
         -0.03579991, -0.02623048],
        [-0.00297978, -0.04134471, -0.03822794, ..., -0.03251299,
          0.04861346, -0.01472353],
        [ 0.04585996,  0.00178183, -0.04419135, ...,  0.0028852 ,
          0.00344677,  0.04910091],
        ...,
        [ 0.03255706,  0.01043719, -0.00717741, ...,  0.03070265,
         -0.03749206, -0.00678954],
        [-0.03119758,  0.01174686, -0.02844997, ...,  0.0138747 ,
          0.03709375, -0.00617547],
        [-0.02554243, -0.02744521, -0.01864302, ..., -0.03729656,
         -0.03260063, -0.0161796 ]]], dtype=float32)>

## Making baseline model : naive bayes

In [ ]:
from os import XATTR_CREATE
model_0 = sklearn.pipeline.Pipeline([
    ("tfidf", sklearn.feature_extraction.text.TfidfVectorizer()),
    ("clf", sklearn.naive_bayes.MultinomialNB())
])

model_0.fit(X_train, Y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [ ]:
preds = model_0.predict(X_test)

In [ ]:
def calc_results(y_true, y_pred):
  model_accuracy = sklearn.metrics.accuracy_score(y_true, y_pred)
  model_precision = sklearn.metrics.precision_score(y_true, y_pred)
  model_recall = sklearn.metrics.recall_score(y_true, y_pred)
  model_f1 = sklearn.metrics.f1_score(y_true, y_pred)
  model_results = {"accuracy": model_accuracy,
                   "precision": model_precision,
                   "recall": model_recall,
                   "f1": model_f1}
  return model_results

In [ ]:
results = calc_results(Y_test,preds)
results

{'accuracy': 0.7926509186351706,
 'precision': 0.8861788617886179,
 'recall': 0.6264367816091954,
 'f1': 0.734006734006734}

## Model 1 A simple dense model

In [ ]:
create_tensorboard_callback("exp", "simple_dense")

Saving TensorBoard log files to: exp/simple_dense/20250113-033637


In [ ]:
inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)
x = txt_vectoriser(inputs)
x = embedding(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
outputs = tf.keras.layers.Dense(1, activation = "sigmoid")(x)
model_1 = tf.keras.Model(inputs, outputs, name = "model_1_dense")

model_1.compile(loss = tf.keras.losses.BinaryCrossentropy(),
                optimizer = tf.keras.optimizers.Adam(),
                metrics = ["accuracy"])

In [ ]:
model_1.fit(x=X_train,
            y = Y_train,
            epochs = 5,
            validation_data = (X_test, Y_test),
            callbacks = [create_tensorboard_callback("exp", "simple_dense")])

Saving TensorBoard log files to: exp/simple_dense/20250113-033637
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.6501 - loss: 0.6475 - val_accuracy: 0.7493 - val_loss: 0.5390
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8128 - loss: 0.4608 - val_accuracy: 0.7822 - val_loss: 0.4758
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8576 - loss: 0.3605 - val_accuracy: 0.7953 - val_loss: 0.4590
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8928 - loss: 0.2860 - val_accuracy: 0.7835 - val_loss: 0.4636
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9173 - loss: 0.2356 - val_accuracy: 0.7769 - val_loss: 0.4806


In [ ]:
model_1.evaluate(X_test, Y_test)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7636 - loss: 0.5105


[0.48064419627189636, 0.7769029140472412]

In [ ]:
preds = model_1.predict(X_test)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [ ]:
preds = tf.squeeze(tf.round(preds))

In [ ]:
preds.numpy()

array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 1., 1., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1., 0.,
       1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 1., 0., 1., 0., 1., 0.,
       1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 0., 1., 1., 1., 0.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 1., 1., 0., 1., 0., 0.,
       0., 0., 0., 1., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0.,
       0., 1., 1., 0., 1., 1., 0., 1., 1., 0., 0., 0., 0., 1., 1., 1., 1.,
       0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 1., 1., 1., 0.,
       1., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 1.,
       0., 0., 1., 1., 0., 1., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0.,
       1., 1., 1., 0., 1., 0., 0., 0., 1., 1., 0., 1., 1., 1., 1., 0., 0.,
       1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 1., 0.,
       0., 1., 1., 1., 1., 1., 0., 1., 0., 0., 0., 0., 1., 1., 1., 0., 1.,
       0., 0., 0., 1., 0.

In [ ]:
Dense_results = calc_results(Y_test, preds.numpy())
Dense_results

{'accuracy': 0.7769028871391076,
 'precision': 0.7947019867549668,
 'recall': 0.6896551724137931,
 'f1': 0.7384615384615385}

## To use embedding projector

In [ ]:
txt_vectoriser.get_vocabulary()[:10]

['', '[UNK]', 'the', 'a', 'in', 'to', 'of', 'and', 'i', 'is']

In [ ]:
embed_weights = model_1.get_layer("embedding").get_weights()[0]

In [ ]:
tf.constant(embed_weights).shape

TensorShape([10000, 128])

In [ ]:
import io
out_v = io.open('vectors.tsv', 'w', encoding='utf-8')
out_m = io.open('metadata.tsv', 'w', encoding='utf-8')

for index, word in enumerate(txt_vectoriser.get_vocabulary()):
  if index == 0:
    continue  # skip 0, it's padding.
  vec = embed_weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + "\n")
  out_m.write(word + "\n")
out_v.close()
out_m.close()

In [ ]:
try:
  from google.colab import files
  files.download('vectors.tsv')
  files.download('metadata.tsv')
except Exception:
  pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Model 2 RNN

### LSTM

In [ ]:
inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)
x = txt_vectoriser(inputs)
x = embedding(x)
x = tf.keras.layers.LSTM(64, return_sequences = True)(x)
x = tf.keras.layers.LSTM(64)(x)
ouputs = tf.keras.layers.Dense(1, activation = "sigmoid")(x)
model_2 = tf.keras.Model(inputs, ouputs, name = "model_2_LSTM")

model_2.compile(loss = tf.keras.losses.BinaryCrossentropy(),
                optimizer = tf.keras.optimizers.Adam(),
                metrics = ["accuracy"])

In [ ]:
model_2.fit(X_train,
            Y_train,
            epochs = 5,
            validation_data = (X_test,Y_test),
            callbacks = create_tensorboard_callback("exp", "LSTM"))

Saving TensorBoard log files to: exp/LSTM/20250113-033649
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9089 - loss: 0.2853 - val_accuracy: 0.7822 - val_loss: 0.5657
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9419 - loss: 0.1574 - val_accuracy: 0.7782 - val_loss: 0.7231
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9544 - loss: 0.1295 - val_accuracy: 0.7769 - val_loss: 0.7979
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9612 - loss: 0.1009 - val_accuracy: 0.7743 - val_loss: 0.9108
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9646 - loss: 0.0861 - val_accuracy: 0.7795 - val_loss: 0.8508


In [ ]:
preds_2 = model_2.predict(X_test)
preds_2 = tf.squeeze(tf.round(preds_2)).numpy()
preds_2

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step


array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 1., 1., 0., 1., 0., 1., 1., 0., 0., 0., 1., 1., 0., 0., 0.,
       1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 1., 1., 1., 0., 1., 0.,
       0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 1., 1., 0., 1., 1., 1., 0.,
       1., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0.,
       0., 1., 1., 0., 1., 1., 0., 1., 1., 0., 0., 0., 0., 0., 1., 1., 1.,
       0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 1., 1., 1., 0.,
       1., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0.,
       0., 0., 1., 1., 0., 1., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0.,
       1., 1., 1., 0., 1., 0., 0., 0., 1., 1., 0., 1., 1., 1., 1., 0., 0.,
       1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 1., 0.,
       0., 1., 1., 1., 1., 1., 0., 1., 0., 0., 0., 0., 1., 1., 1., 0., 1.,
       0., 0., 0., 1., 0.

In [ ]:
calc_results(Y_test, preds_2)

{'accuracy': 0.7795275590551181,
 'precision': 0.7941176470588235,
 'recall': 0.6982758620689655,
 'f1': 0.7431192660550459}

## Model 3: GRU

In [ ]:
inputs = tf.keras.layers.Input(shape=(1,), dtype = tf.string)
x = txt_vectoriser(inputs)
x = embedding(x)
x = tf.keras.layers.GRU(64, return_sequences = True)(x)
x = tf.keras.layers.LSTM(64, return_sequences=True)(x)
x = tf.keras.layers.GRU(64)(x)
x = tf.keras.layers.Dense(64, activation = "relu")(x)
outputs = tf.keras.layers.Dense(1, activation = "sigmoid")(x)
model_3 = tf.keras.Model(inputs, outputs, name = "model_3_GRU")

In [ ]:
model_3.compile(loss = tf.keras.losses.BinaryCrossentropy(),
                optimizer = tf.keras.optimizers.Adam(),
                metrics = ["accuracy"])

In [ ]:
model_3.fit(X_train,
          Y_train,
          epochs = 5,
          validation_data = (X_test, Y_test),
          callbacks = create_tensorboard_callback("exp", "GRU"))

Saving TensorBoard log files to: exp/GRU/20250113-033706
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9522 - loss: 0.2201 - val_accuracy: 0.7677 - val_loss: 1.0091
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9710 - loss: 0.0718 - val_accuracy: 0.7808 - val_loss: 0.9287
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9715 - loss: 0.0626 - val_accuracy: 0.7730 - val_loss: 1.0809
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9727 - loss: 0.0598 - val_accuracy: 0.7612 - val_loss: 1.4294
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9753 - loss: 0.0510 - val_accuracy: 0.7703 - val_loss: 1.3356


In [ ]:
preds_GRU = model_3.predict(X_test)
preds_GRU = tf.squeeze(tf.round(preds_GRU)).numpy()

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step


In [ ]:
calc_results(Y_test, preds_GRU)

{'accuracy': 0.7703412073490814,
 'precision': 0.7582089552238805,
 'recall': 0.7298850574712644,
 'f1': 0.7437774524158126}

## Model 4 Bidirectional RNN

In [ ]:
inputs = tf.keras.layers.Input(shape=(1,), dtype = tf.string)
x = txt_vectoriser(inputs)
x = embedding(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences = True))(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64))(x)
x = tf.keras.layers.Dense(64, activation = "relu")(x)
outputs = tf.keras.layers.Dense(1, activation = "sigmoid")(x)
model_4 = tf.keras.Model(inputs, outputs, name = "model_4_Bidirectional")

In [ ]:
model_4.compile(loss = tf.keras.losses.BinaryCrossentropy(),
                optimizer = tf.keras.optimizers.Adam(),
                metrics = ["accuracy"])

In [ ]:
model_4.fit(X_train,
            Y_train,
            epochs = 5,
            validation_data = (X_test, Y_test),
            callbacks = [create_tensorboard_callback("exp", "Bidirectional")])

Saving TensorBoard log files to: exp/Bidirectional/20250113-033726
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.9525 - loss: 0.1817 - val_accuracy: 0.7848 - val_loss: 1.0105
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9765 - loss: 0.0519 - val_accuracy: 0.7651 - val_loss: 1.6078
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9781 - loss: 0.0456 - val_accuracy: 0.7795 - val_loss: 1.8102
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9820 - loss: 0.0398 - val_accuracy: 0.7690 - val_loss: 1.6186
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9794 - loss: 0.0442 - val_accuracy: 0.7822 - val_loss: 1.6231


In [ ]:
model_bi_preds = model_4.predict(X_test)
model_bi_preds = tf.squeeze(tf.round(model_bi_preds)).numpy()

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step


In [ ]:
calc_results(Y_test, model_bi_preds)

{'accuracy': 0.7821522309711286,
 'precision': 0.8137931034482758,
 'recall': 0.6781609195402298,
 'f1': 0.7398119122257053}

## Model 5: Conv1D

In [ ]:
inputs = tf.keras.layers.Input(shape=(1,), dtype = tf.string)
x = txt_vectoriser(inputs)
x = embedding(x)
x = tf.keras.layers.Conv1D(filters = 64, kernel_size = 5, activation = "relu")(x)
x = tf.keras.layers.GlobalMaxPool1D()(x)
outputs = tf.keras.layers.Dense(1, activation = "sigmoid")(x)
model_5 = tf.keras.Model(inputs, outputs)

In [ ]:
model_5.compile(loss = tf.keras.losses.BinaryCrossentropy(),
                optimizer = tf.keras.optimizers.Adam(),
                metrics = ["accuracy"])

In [ ]:
model_5.fit(X_train,
            Y_train,
            epochs = 5,
            validation_data = (X_test, Y_test),
            callbacks = [create_tensorboard_callback("exp", "Conv1D")])

Saving TensorBoard log files to: exp/Conv1D/20250113-033748
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9328 - loss: 0.1942 - val_accuracy: 0.7730 - val_loss: 0.8733
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9742 - loss: 0.0730 - val_accuracy: 0.7743 - val_loss: 0.9799
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9777 - loss: 0.0620 - val_accuracy: 0.7664 - val_loss: 1.0575
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9807 - loss: 0.0525 - val_accuracy: 0.7612 - val_loss: 1.1397
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9803 - loss: 0.0494 - val_accuracy: 0.7585 - val_loss: 1.1799


In [ ]:
model_conv_preds = model_5.predict(X_test)
model_conv_preds = tf.squeeze(tf.round(model_conv_preds)).numpy()

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


In [ ]:
calc_results(Y_test, model_conv_preds)

{'accuracy': 0.7585301837270341,
 'precision': 0.7611464968152867,
 'recall': 0.6867816091954023,
 'f1': 0.7220543806646526}